# Построение ML-модели классификации на основании подготовленного ранее датасета.

## Цель:
Цель исследования — научиться эффективно прогнозировать прекращение сотрудничества продавцов с нашей платформой продаж и доставки товаров. Учитывая, что пользователями нашего сервиса выступают обе стороны рынка — как покупатели, так и продавцы, важно отметить, что задачей текущего проекта является не отслеживание оттока покупателей от отдельных продавцов, а выявление факторов риска ухода самих продавцов из платформы в целом. Для реализации поставленной цели воспользуемся специально подготовленным набором данных, содержащим информацию о продавцах, давно прекративших свою активность, а также о тех, кто длительное время находился без активных сделок перед наступлением аналогичного состояния.

## Выводы по итогам экспериментов
Проведённые эксперименты позволили оценить способность различных моделей предугадывать вероятность ухода клиентов-продавцов с платформы.

Логистическая регрессия (LogisticRegression)
Показала высокое значение precision (точность равна 1.0), однако чрезвычайно низкий уровень recall (всего 0.07), что свидетельствует о слабой способности обнаруживать истинные позитивные события. Низкое значение F1 подтверждает недостаточную пригодность данной модели для поставленных целей.

Нейронная сеть (MLPClassifier)
Первая версия нейронной сети продемонстрировала хороший balance между точностью и полнотой, хотя точность слегка уступала единице (precision ≈ 0.95), а full-recall составил 0.68. Значение F1 оказалось приемлемым (0.79), однако существовала тенденция к переобучению.

Случайный лес (RandomForestClassifier)
Этот ансамбль показал почти идеальный balance между точностью и полнотой: precision равен 1.0, а recall составляет 0.86. Высокий показатель F1 приближён к 0.92, демонстрируя хорошие перспективы для практического применения.

Градиентный бустинг (GradientBoostingClassifier)
Эта модель превзошла предыдущие результаты, показывая отличную точность (precision = 1.0) и высокий recall (≈0.93). Благодаря этому достигнут самый высокий показатель F1 — 0.96, подчёркивая стабильность и надёжность градиентного бустинга.

Улучшенная нейронная сеть (MLPClassifier)
Последняя попытка с нейронной сетью привела к отличным показателям: высокой точности (≈0.97) и абсолютному уровню полноты (recall = 1.0). F1-мера составила впечатляющие 0.98, подтвердив потенциал нейронных сетей для задач прогнозирования поведения пользователей.
Инструменты подбора гиперпараметров

Экспериментально подтверждено, что разница между инструментами подбора гиперпараметров, такими как RandomizedSearchCV и GridSearchCV, оказалась незначительной в данном конкретном сценарии. Оба подхода показали схожие результаты, и выбор одного из них не оказал решающего влияния на итоговую производительность моделей.

Рекомендации по дальнейшим действиям
Из полученных результатов наиболее перспективными моделями выглядят случайный лес и градиентный бустинг. Эти модели демонстрируют стабильно высокие показатели во всех аспектах и подходят для внедрения в продакшен. Дальнейшие шаги включают улучшение используемых признаков, углубленную диагностику данных и возможное расширение размера обучающей выборки для предотвращения переобучения.

In [ ]:
# Основные библиотеки
import pandas as pd
import numpy as np
from collections import Counter 

# Разделение выборки
from sklearn.model_selection import train_test_split, GridSearchCV

# Метрики качества модели
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier



In [60]:
## Общая таблица для построения модели.
aggregated_df = pd.read_csv('train_test.csv')
aggregated_df.columns

Index(['Unnamed: 0', 'orders_first_30_days', 'freight_value_first_30_days',
       'orders_second_30_days', 'freight_value_second_30_days',
       'orders_last_30_days', 'freight_value_last_30_days',
       'avg_review_score_first_30_days', 'avg_price_first_30_days',
       'avg_review_score_second_30_days', 'avg_price_second_30_days',
       'avg_review_score_last_30_days', 'avg_price_last_30_days',
       'lost_seller', 'delta_orders_second', 'delta_orders_first',
       'delta_score_second', 'delta_score_first', 'delta_freight_first',
       'delta_freight_second', 'delta_price_first', 'delta_price_second'],
      dtype='object')

In [61]:
target_column = 'lost_seller'
X = aggregated_df.drop(target_column, axis=1)
y = aggregated_df[target_column]

# Разделяем на обучающую и тестовую выборку с соотношением 4:1 (test_size=0.2)
# Stratify=y позволяет сохранять баланс классов
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
# Параметры для поиска (ограничим их, чтобы избежать переобучения)
param_distribution = {
    'n_estimators': [3, 6, 9],  #  количество деревьев
    'max_depth': [5, 10, 15],         #  глубину деревьев
    'min_samples_split': [2, 4, 6],  #  минимальное количество образцов для расщепления
    'min_samples_leaf': [2, 4, 6],  #  минимальное количество образцов в листьях
    'bootstrap': [True, False]
}

# Модель
rf = RandomForestClassifier(random_state=42)

# Randomized Search CV
rand_search = RandomizedSearchCV(estimator=rf, param_distributions=param_distribution, n_iter=100, cv=5, scoring='f1', verbose=2, random_state=42, n_jobs=-1)
rand_search.fit(X_train, y_train)

# Лучшие параметры
best_params = rand_search.best_params_
print("Best Parameters:", best_params)

# Лучшая модель
best_rf = rand_search.best_estimator_

# Прогнозы на тестовой выборке
predictions = best_rf.predict(X_test)

# Оценка точности
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions)
print("F1 Score:", f1)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'n_estimators': 6, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 15, 'bootstrap': True}
Accuracy: 0.9991850040749797
Precision: 1.0
Recall: 0.8571428571428571
F1 Score: 0.9230769230769231


In [80]:
# Параметры для поиска (ограничим их, чтобы избежать переобучения)
param_distribution = {
    'n_estimators': [3, 6, 9],  #  количество деревьев
    'max_depth': [5, 10, 15],         #  глубину деревьев
    'min_samples_split': [2, 4, 6],  #  минимальное количество образцов для расщепления
    'min_samples_leaf': [2, 4, 6],  #  минимальное количество образцов в листьях
    'bootstrap': [True, False]
}

# Модель
rf = RandomForestClassifier(random_state=42)

# Grid Search CV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='f1', verbose=2, n_jobs=-1)
rand_search.fit(X_train, y_train)

# Лучшие параметры
best_params = rand_search.best_params_
print("Best Parameters:", best_params)

# Лучшая модель
best_rf = rand_search.best_estimator_

# Прогнозы на тестовой выборке
predictions = best_rf.predict(X_test)

# Оценка точности
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions)
print("F1 Score:", f1)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'n_estimators': 6, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 15, 'bootstrap': True}
Accuracy: 0.9991850040749797
Precision: 1.0
Recall: 0.8571428571428571
F1 Score: 0.9230769230769231


In [84]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Параметры для поиска
param_distribution = {
    'hidden_layer_sizes': [(2,2), (4,4)],
    # 'activation': ['identity', 'logistic', 'tanh', 'relu'],
    'solver': ['adam'], # 'sgd', 'adam'  lbfgs
    'alpha': [0.01],
    'learning_rate': ['constant'] # , 'invscaling', 'adaptive'
}

# Модель
mlp = MLPClassifier(max_iter=1000, random_state=42)

# Randomized Search CV
rand_search_mlp = RandomizedSearchCV(estimator=mlp, param_distributions=param_distribution, n_iter=100, cv=5, scoring='f1', verbose=2, random_state=42, n_jobs=-1)
rand_search_mlp.fit(X_train_scaled, y_train)

# Лучшие параметры
best_params_mlp = rand_search_mlp.best_params_
print("Best Parameters:", best_params_mlp)

# Лучшая модель
best_mlp = rand_search_mlp.best_estimator_

# Прогнозы на тестовой выборке
predictions_mlp = best_mlp.predict(X_test_scaled)

# Оценка точности
accuracy = accuracy_score(y_test, predictions_mlp)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions_mlp)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions_mlp)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions_mlp)
print("F1 Score:", f1)

c:\Users\днс\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=100. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best Parameters: {'solver': 'adam', 'learning_rate': 'constant', 'hidden_layer_sizes': (4, 4), 'alpha': 0.01}
Accuracy: 0.9979625101874491
Precision: 0.95
Recall: 0.6785714285714286
F1 Score: 0.7916666666666666


In [88]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Параметры для поиска
param_distribution = {
    'hidden_layer_sizes': [(3,3), (5,3)],
    # 'activation': ['identity', 'logistic', 'tanh', 'relu'],
    'solver': ['lbfgs'], # 'sgd', 'adam'  lbfgs
    'alpha': [0.001],
    'learning_rate': ['constant'] # , 'invscaling', 'adaptive'
}

# Модель
mlp = MLPClassifier(max_iter=1000, random_state=42)

# Randomized Search CV
rand_search_mlp = RandomizedSearchCV(estimator=mlp, param_distributions=param_distribution, n_iter=100, cv=5, scoring='f1', verbose=2, random_state=42, n_jobs=-1)
rand_search_mlp.fit(X_train_scaled, y_train)

# Лучшие параметры
best_params_mlp = rand_search_mlp.best_params_
print("Best Parameters:", best_params_mlp)

# Лучшая модель
best_mlp = rand_search_mlp.best_estimator_

# Прогнозы на тестовой выборке
predictions_mlp = best_mlp.predict(X_test_scaled)

# Оценка точности
accuracy = accuracy_score(y_test, predictions_mlp)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions_mlp)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions_mlp)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions_mlp)
print("F1 Score:", f1)

c:\Users\днс\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=100. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best Parameters: {'solver': 'lbfgs', 'learning_rate': 'constant', 'hidden_layer_sizes': (5, 3), 'alpha': 0.001}
Accuracy: 0.9997962510187449
Precision: 0.9655172413793104
Recall: 1.0
F1 Score: 0.9824561403508771


In [ ]:
# Параметры для поиска
param_distribution = {
    'learning_rate': [0.01, 0.1],
    # 'n_estimators': [5, 10],
    'max_depth': [1, 2],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 3],
    'subsample': [0.8, 0.9]
}

# Модель
gbc = GradientBoostingClassifier(random_state=42)

# Randomized Search CV
rand_search_gbc = RandomizedSearchCV(estimator=gbc, param_distributions=param_distribution, n_iter=100, cv=5, scoring='f1', verbose=2, random_state=42, n_jobs=-1)
rand_search_gbc.fit(X_train, y_train)

# Лучшие параметры
best_params_gbc = rand_search_gbc.best_params_
print("Best Parameters:", best_params_gbc)

# Лучшая модель
best_gbc = rand_search_gbc.best_estimator_

# Прогнозы на тестовой выборке
predictions_gbc = best_gbc.predict(X_test)

# Оценка точности
accuracy = accuracy_score(y_test, predictions_gbc)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions_gbc)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions_gbc)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions_gbc)
print("F1 Score:", f1)

c:\Users\днс\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 32 is smaller than n_iter=100. Running 32 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best Parameters: {'subsample': 0.8, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 2, 'learning_rate': 0.1}
Accuracy: 0.9995925020374898
Precision: 1.0
Recall: 0.9285714285714286
F1 Score: 0.9629629629629629


In [92]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Расширенный набор параметров для поиска
param_distribution = {
    'penalty': ['l2'],  # Тип регуляризации
    'solver': ['newton-cg', 'sag', 'saga'],  # Добавляем дополнительные оптимизаторы
    'C': np.logspace(-8, 8, 50),            # Увеличиваем диапазон коэффициентов регуляризации
    'fit_intercept': [True, False],        # Включаем свободный член
    'class_weight': ['balanced', None],     # Балансировка классов
    'multi_class': ['ovr'], # Методы многоклассовой классификации
    'max_iter': [1000, 10000]          # Больше итераций для улучшения качества обучения
}

# Модель
logreg = LogisticRegression(random_state=42)

# Randomized Search CV
rand_search_logreg = RandomizedSearchCV(estimator=logreg, param_distributions=param_distribution, n_iter=100, cv=5, scoring='f1', verbose=2, random_state=42, n_jobs=-1)
rand_search_logreg.fit(X_train_scaled, y_train)

# Лучшие параметры
best_params_logreg = rand_search_logreg.best_params_
print("Best Parameters:", best_params_logreg)

# Лучшая модель
best_logreg = rand_search_logreg.best_estimator_

# Прогнозы на тестовой выборке
predictions_logreg = best_logreg.predict(X_test_scaled)

# Оценка точности
accuracy = accuracy_score(y_test, predictions_logreg)
print("Accuracy:", accuracy)

# Оценка precision
precision = precision_score(y_test, predictions_logreg)
print("Precision:", precision)

# Оценка recall
recall = recall_score(y_test, predictions_logreg)
print("Recall:", recall)

# Оценка F1-Score
f1 = f1_score(y_test, predictions_logreg)
print("F1 Score:", f1)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


c:\Users\днс\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Best Parameters: {'solver': 'sag', 'penalty': 'l2', 'multi_class': 'ovr', 'max_iter': 10000, 'fit_intercept': True, 'class_weight': None, 'C': np.float64(10481131.341546832)}
Accuracy: 0.9947025264873676
Precision: 1.0
Recall: 0.07142857142857142
F1 Score: 0.13333333333333333
